<a href="https://colab.research.google.com/github/sunitharamu-1983/GENAI-Learning-Journey/blob/main/Quiz_Gen_Ollama.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Quiz from Meeting Summary

### Open Source Ollama Model

Models

To bypass Gemini API quotas and use open-source models directly within Colab, we'll install and configure Ollama. Ollama acts as a local server that allows you to run various models like Llama 3, Mistral, and others. We'll then update our `generate_quiz` function to use this local Ollama instance.

In [5]:
# Install zstd (required for Ollama installation script)
!sudo apt-get update && sudo apt-get install -y zstd

# Install Ollama (server and client library)
!curl -fsSL https://ollama.com/install.sh | sh
!pip install ollama

print("Ollama and its Python client library installed.")

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [93.4 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,615 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.2 MB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,294 kB]
Get:13 http://security.ubuntu.com/u

In [7]:
import os
import json
import ollama # Make sure ollama is imported here if not already
from typing import List, Dict

# -------------------------------
# 1. Generate multiple-choice questions from transcript
# -------------------------------
# Refactored generate_quiz to accept an Ollama client
def generate_quiz_ollama(ollama_client, transcript_text: str, model_name: str = 'phi3', temperature: float = 0.3, num_questions: int = 5) -> List[Dict]:
    """
    Sends the transcript to an Ollama model and asks for a JSON list of MCQs.
    Each question dict has: question, options (list of 4), answer (0-indexed).
    """
    prompt = f"""You are an AI that creates multiple-choice quizzes from meeting transcripts.
Based on the transcript below, generate exactly {num_questions} multiple-choice questions.
Each question must test understanding of key points, decisions, or action items.

Return ONLY valid JSON in the following format (no extra text):

[
  {{
    "question": "What was the main decision about X?",
    "options": ["Option A", "Option B", "Option C", "Option D"],
    "answer": 0
  }},
  ...
]

The "answer" field is the index (0-based) of the correct option.
Transcript:
{transcript_text[:12000]}   # limit to avoid context overflow
"""

    try:
        response = ollama_client.chat(
            model=model_name,
            messages=[
                {"role": "user", "content": prompt}
            ],
            options={
                "temperature": temperature
            }
        )
        # Ollama's response.message.content contains the string output
        raw = response['message']['content'].strip()
        questions = json.loads(raw)
        return questions[:num_questions]   # ensure we have exactly the requested number
    except Exception as e:
        print(f"Error generating quiz with Ollama: {e}")
        return []

# -------------------------------
# 2. Run the quiz interactively and score (reusing the existing function)
# -------------------------------
def run_quiz(questions: List[Dict]) -> None:
    if not questions:
        print("No questions generated. Cannot run quiz.")
        return

    print("\n" + "="*50)
    print("📝 QUIZ TIME! Answer the following multiple-choice questions.")
    print("Enter the letter of your answer (A, B, C, or D).")
    print("="*50 + "\n")

    score = 0
    for i, q in enumerate(questions, start=1):
        print(f"Question {i}: {q['question']}")
        for idx, opt in enumerate(q['options']):
            print(f"   {chr(65+idx)}. {opt}")

        # Get user input with validation
        while True:
            answer_letter = input("\nYour answer (A/B/C/D): ").strip().upper()
            if answer_letter in ['A','B','C','D']:
                break
            print("Please enter A, B, C, or D.")

        correct_index = q['answer']
        correct_letter = chr(65 + correct_index)
        if answer_letter == correct_letter:
            print("✅ Correct!\n")
            score += 1
        else:
            print(f"❌ Incorrect. The correct answer was {correct_letter}. {q['options'][correct_index]}\n")

    print("="*50)
    print(f"🎯 Your final score: {score} / {len(questions)} ({score/len(questions)*100:.1f}%))")
    if score == len(questions):
        print("Perfect! You really understood the meeting.")
    elif score >= len(questions)*0.7:
        print("Good job! You have a solid grasp.")
    else:
        print("You might want to review the transcript again.")
    print("="*50)

In [9]:
import subprocess
import time
import ollama
import os

# Ensure /usr/local/bin is in the PATH for shell commands if it's not already
%env PATH=$PATH:/usr/local/bin

# Start Ollama server in the background using nohup and the full path
print("Starting Ollama server in background...")
# Use /tmp/ollama.log for easier access and ensure it's written
!/usr/bin/nohup /usr/local/bin/ollama serve > /tmp/ollama.log 2>&1 &

# Give Ollama some initial time to start up
print("Giving server time to initialize...")
time.sleep(25) # Increased initial sleep time further to be safe

# Polling loop to wait for Ollama server to be ready via API
print("Waiting for Ollama server to be responsive via API...")
max_retries = 20 # Increased retries
for i in range(max_retries):
    try:
        ollama_client_check = ollama.Client(host='http://127.0.0.1:11434')
        ollama_client_check.list() # A simple call to check connectivity
        print("Ollama server is responsive.")
        break
    except ollama.ResponseError as e:
        print(f"Attempt {i+1}/{max_retries}: Ollama server not yet responsive (ResponseError: {e}). Retrying in 5 seconds...")
        time.sleep(5)
    except Exception as e:
        print(f"Attempt {i+1}/{max_retries}: Ollama server not yet responsive (General Error: {e}). Retrying in 5 seconds...")
        time.sleep(5)
else:
    print("ERROR: Failed to connect to Ollama server after multiple retries. Check /tmp/ollama.log for errors.")
    # Attempt to print log contents again if it failed
    if os.path.exists('/tmp/ollama.log'):
        with open('/tmp/ollama.log', 'r') as f:
            print("--- Final Ollama Log Content ---")
            print(f.read())
            print("--------------------------------")
    raise RuntimeError("Ollama server did not become responsive after multiple retries.") # Explicitly raise error

# Pull a smaller model (e.g., phi3) to speed up execution
print("Pulling the phi3 model...")
!/usr/local/bin/ollama pull phi3

# Initialize the Ollama client for general use
ollama_client = ollama.Client(host='http://127.0.0.1:11434')

print("Ollama server started and phi3 model pulled.")

env: PATH=$PATH:/usr/local/bin
Starting Ollama server in background...
Giving server time to initialize...
Waiting for Ollama server to be responsive via API...
Ollama server is responsive.
Pulling the phi3 model...

Ollama server started and phi3 model pulled.


In [12]:
import os
import json
import ollama # Make sure ollama is imported here if not already
import re # Import regex module
from typing import List, Dict

# -------------------------------
# 1. Generate multiple-choice questions from transcript
# -------------------------------
# Refactored generate_quiz to accept an Ollama client
def generate_quiz_ollama(ollama_client, transcript_text: str, model_name: str = 'phi3', temperature: float = 0.3, num_questions: int = 5) -> List[Dict]:
    """
    Sends the transcript to an Ollama model and asks for a JSON list of MCQs.
    Each question dict has: question, options (list of 4), answer (0-indexed).
    """
    prompt = f"""You are an AI that creates multiple-choice quizzes from meeting transcripts.
Based on the transcript below, generate exactly {num_questions} multiple-choice questions.
Each question must test understanding of key points, decisions, or action items.

Return ONLY valid JSON in the following format (no extra text):

[
  {{
    "question": "What was the main decision about X?",
    "options": ["Option A", "Option B", "Option C", "Option D"],
    "answer": 0
  }},
  ...
]

The "answer" field is the index (0-based) of the correct option.
Transcript:
{transcript_text[:12000]}   # limit to avoid context overflow
"""

    try:
        response = ollama_client.chat(
            model=model_name,
            messages=[
                {"role": "user", "content": prompt}
            ],
            options={
                "temperature": temperature
            }
        )
        # Ollama's response.message.content contains the string output
        raw = response['message']['content'].strip()

        # Use regex to extract only the JSON part from the response
        # This is more robust in case the LLM adds introductory/concluding text
        json_match = re.search(r'\s*(\[[\s\S]*\])\s*', raw)
        if json_match:
            json_string = json_match.group(1)
            questions = json.loads(json_string)
        else:
            # If regex fails to find clear JSON, try parsing the raw string anyway
            # This can happen if the LLM output is very messy or not bracketed.
            print(f"Warning: Could not extract JSON using regex. Attempting to parse raw response.\nRaw Response: {raw[:500]}...")
            questions = json.loads(raw)

        return questions[:num_questions]   # ensure we have exactly the requested number
    except json.JSONDecodeError as e:
        print(f"Error decoding JSON from Ollama response: {e}")
        print(f"Problematic JSON snippet: {raw[e.pos-50 : e.pos+50] if hasattr(e, 'pos') else raw[:100]}...")
        return []
    except Exception as e:
        print(f"Error generating quiz with Ollama: {e}")
        return []

# -------------------------------
# 2. Run the quiz interactively and score (reusing the existing function)
# -------------------------------
def run_quiz(questions: List[Dict]) -> None:
    if not questions:
        print("No questions generated. Cannot run quiz.")
        return

    print("\n" + "="*50)
    print("📝 QUIZ TIME! Answer the following multiple-choice questions.")
    print("Enter the letter of your answer (A, B, C, or D).")
    print("="*50 + "\n")

    score = 0
    for i, q in enumerate(questions, start=1):
        print(f"Question {i}: {q['question']}")
        for idx, opt in enumerate(q['options']):
            print(f"   {chr(65+idx)}. {opt}")

        # Get user input with validation
        while True:
            answer_letter = input("\nYour answer (A/B/C/D): ").strip().upper()
            if answer_letter in ['A','B','C','D']:
                break
            print("Please enter A, B, C, or D.")

        correct_index = q['answer']
        correct_letter = chr(65 + correct_index)
        if answer_letter == correct_letter:
            print("✅ Correct!\n")
            score += 1
        else:
            print(f"❌ Incorrect. The correct answer was {correct_letter}. {q['options'][correct_index]}\n")

    print("="*50)
    print(f"🎯 Your final score: {score} / {len(questions)} ({score/len(questions)*100:.1f}%))")
    if score == len(questions):
        print("Perfect! You really understood the meeting.")
    elif score >= len(questions)*0.7:
        print("Good job! You have a solid grasp.")
    else:
        print("You might want to review the transcript again.")
    print("="*50)

### Run Quiz with Ollama
Now, let's run the quiz generation and interaction using the Ollama client and the downloaded transcript.

In [13]:
import os # Import the os module here

# Check if input_file_path is defined, if not, download the specified transcript.
# This ensures the quiz generation can proceed even if the transcript download cell was skipped.
if 'input_file_path' not in locals() and 'input_file_path' not in globals():
    print("Downloading the specified transcript as 'input_file_path' was not defined...")
    # Using the publicly available transcript from the provided URL
    !wget https://raw.githubusercontent.com/sunitharamu-1983/GENAI-Learning-Journey/main/01_Foundations/Session%2011/Meeting_Summary.md -O meeting_summary.md
    input_file_path = "meeting_summary.md"
    print(f"Transcript downloaded to: {input_file_file_path}")

if input_file_path and os.path.exists(input_file_path):
    with open(input_file_path, "r", encoding="utf-8") as f:
        transcript = f.read()

    print("\n🧠 Generating multiple-choice questions from the transcript using Ollama (phi3)...")
    # Use the new generate_quiz_ollama function with the phi3 model
    quiz_questions_ollama = generate_quiz_ollama(ollama_client, transcript, model_name='phi3', temperature=0.3, num_questions=5)

    if quiz_questions_ollama:
        run_quiz(quiz_questions_ollama)
    else:
        print("Could not generate quiz with Ollama. Check your Ollama server and model configuration.")
else:
    print("Cannot proceed with quiz generation: transcript file not found or could not be downloaded.")


🧠 Generating multiple-choice questions from the transcript using Ollama (phi3)...

📝 QUIZ TIME! Answer the following multiple-choice questions.
Enter the letter of your answer (A, B, C, or D).

Question 1: What is the main purpose behind giving away open-source LLMs for free?
   A. Attract talent and researchers.
   B. Build community → later sell a proprietary model.
   C. Get investors by showing large user base.

Your answer (A/B/C/D): A
✅ Correct!

Question 2: What is the typical size reduction in memory when quantizing an Olama-compatible LLM?
   A. From ~54 GB to ~17 GB for a model with about 2 billion parameters.
   B. No significant change.

Your answer (A/B/C/D): A
✅ Correct!

Question 3: What does the term 'quantization' refer to in machine learning?
   A. A one-time process that reduces model size at a small accuracy cost.
   B. An ongoing training method for large models.

Your answer (A/B/C/D): A
✅ Correct!

Question 4: Which tool is recommended if you have limited RAM bu